# Робота з індексом стану рослинності

Тут реалізовані необхідні дії з даними, які було з сайту наукового центру NESDIS STAR

Створити віртуальне середовище (venv) в якому будуть встановлені всі необхідні бібліотеки та налаштування для даної лабораторної роботи

Ініціалізуємо необхідні бібліотеки, змінні та словарі

In [16]:
from urllib.request import urlopen
import time
import os
import re
import glob
import pandas as pd
import numpy as np

parsed_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={province_id}&year1=1981&year2=2026&type=Mean"
province_ids = list(range(1,28))
province_names = {
    1: "Cherkasy",
    2: "Chernihiv",
    3: "Chernivtsi",
    4: "Crimea",
    5: "Dnipropetrovs'k",
    6: "Donets'k",
    7: "Ivano-Frankivs'k",
    8: "Kharkiv",
    9: "Kherson",
    10: "Khmel'nyts'kyy",
    11: "Kiev",
    12: "Kiev City",
    13: "Kirovohrad",
    14: "Luhans'k",
    15: "L'viv",
    16: "Mykolayiv",
    17: "Odessa",
    18: "Poltava",
    19: "Rivne",
    20: "Sevastopol'",
    21: "Sumy",
    22: "Ternopil'",
    23: "Transcarpathia",
    24: "Vinnytsya",
    25: "Volyn",
    26: "Zaporizhzhya",
    27: "Zhytomyr"
}
changer_dict = {
        1: 25,
        2: 27,
        3: 26,
        4: 1,
        5: 4,
        6: 5,
        7: 9,
        8: 22,
        9: 23,
        10: 24,
        11: 11,
        12: 10,
        13: 12,
        14: 13,
        15: 14,
        16: 15,
        17: 16,
        18: 17,
        19: 18,
        20: 19,
        21: 20,
        22: 21,
        23: 7,
        24: 2,
        25: 3,
        26: 8,
        27: 6
    }
download_dir = "./data"
os.makedirs(download_dir, exist_ok=True)

Для кожної з адміністративних одиниць України завантажити (urllib) тестові структуровані файли, що містять значення VHI-індексу. При зберіганні файлу, до його імені потрібно додати дату та час завантаження. Передбачити повторні запуски скрипту, реалізувати механізм запобігання повторного довантаження та колізії даних.

In [17]:
def file_already_exists(province_id):
    pattern = os.path.join(download_dir,f"vhi_province_{province_id}_1981_2026_Mean_*.csv")
    files = glob.glob(pattern)
    return len(files) > 0

def download_file():
    for province_id in province_ids:
        if file_already_exists(province_id):
            continue

        url = parsed_url.format(province_id=province_id)

        current_time = time.strftime("%d-%m-%Y_%H-%M-%S")
        destination = os.path.join(download_dir,f"vhi_province_{province_id}_1981_2026_Mean_{current_time}.csv")

        response = urlopen(url)
        text = response.read().decode("utf-8")

        clean_text = re.sub(r"<.*?>", "", text)

        with open(destination, "w", encoding="utf-8") as f:
            f.write(clean_text)

download_file()

Зчитати завантажені текстові файли у pandas dataframe. Здійснити data cleaning: прибрати зайві стовпці, заповнити пропуски, видалити зайвий текст тощо. Додати стовпчики з назвою та індексом області.

In [18]:
def csv_to_df():
    files = glob.glob("data/*.csv")

    dfs = []

    for file in files:
        df = pd.read_csv(
            file,
            skiprows=2,
            header=None,
            names=["year", "week", "SMN", "SMT", "VCI", "TCI", "VHI"],
            usecols=[0, 1, 2, 3, 4, 5, 6],
            skipinitialspace=True
        )

        df.drop(columns=["SMN", "SMT", "VCI", "TCI"], inplace=True)

        df.replace(-1, np.nan, inplace=True)
        df.dropna(inplace=True)

        province_id = int(re.search(r'vhi_province_(\d+)', file).group(1))

        df["province_id"] = province_id
        df["province_name"] = province_names[province_id]

        dfs.append(df)

    return(pd.concat(dfs, ignore_index=True))

df = csv_to_df()
df

,year,week,VHI,province_id,province_name
0,1982,1,49.95,10,Khmel'nyts'kyy
1,1982,2,47.04,10,Khmel'nyts'kyy
2,1982,3,44.99,10,Khmel'nyts'kyy
3,1982,4,41.29,10,Khmel'nyts'kyy
4,1982,5,37.72,10,Khmel'nyts'kyy
...,...,...,...,...,...
60691,2026,6,48.58,9,Kherson
60692,2026,7,49.48,9,Kherson
60693,2026,8,48.88,9,Kherson
60694,2026,9,47.95,9,Kherson


Реалізувати процедуру зміни індексів: в завантажених з NOAA даних області індексуються за англійською абеткою (Province 1 - Cherkasy), потрібно замінити індекси так, щоб області індексувалася за українською абеткою (1 область - Вінницька).

In [23]:
def change_indexes(df):
    df["province_id"] = df["province_id"].map(changer_dict)

change_indexes(df)
df

,year,week,VHI,province_id,province_name
0,1982,1,49.95,2,Khmel'nyts'kyy
1,1982,2,47.04,2,Khmel'nyts'kyy
2,1982,3,44.99,2,Khmel'nyts'kyy
3,1982,4,41.29,2,Khmel'nyts'kyy
4,1982,5,37.72,2,Khmel'nyts'kyy
...,...,...,...,...,...
60691,2026,6,48.58,7,Kherson
60692,2026,7,49.48,7,Kherson
60693,2026,8,48.88,7,Kherson
60694,2026,9,47.95,7,Kherson


Реалізувати процедури для формування вибірок:

Ряд VHI для області за вказаний рік

In [24]:
def filter_province_year(df, province_id, year):
    return(df[(df["province_id"] == province_id) & (df["year"] == year)])

year = int(input("Введіть рік для виборки: "))
province = int(input("Введіть індекс області для виборки: "))
filter_province_year(df, province, year)

,year,week,VHI,province_id,province_name
53526,2018,1,42.06,4,Donets'k
53527,2018,2,44.15,4,Donets'k
53528,2018,3,45.70,4,Donets'k
53529,2018,4,46.47,4,Donets'k
53530,2018,5,45.76,4,Donets'k
53531,2018,6,44.72,4,Donets'k
53532,2018,7,43.49,4,Donets'k
53533,2018,8,42.75,4,Donets'k
53534,2018,9,42.49,4,Donets'k
53535,2018,10,42.86,4,Donets'k


Ряд VHI за вказаний діапазон років для вказаних областей

In [25]:
def filter_provinces_years(df, provinces_ids, years):
    return df[(df["province_id"].isin(provinces_ids)) & (df["year"].isin(years))]

years = list(map(int, input("Введіть роки для виборки у форматі (yyyy yyyy yyyy): ").split()))
provinces = list(map(int, input("Введіть індекси областей для виборки у форматі (ii ii ii): ").split()))
filter_provinces_years(df, provinces, years)

,year,week,VHI,province_id,province_name
7201,1991,1,34.20,10,Kirovohrad
7202,1991,2,34.54,10,Kirovohrad
7203,1991,3,35.38,10,Kirovohrad
7204,1991,4,36.62,10,Kirovohrad
7205,1991,5,34.71,10,Kirovohrad
...,...,...,...,...,...
53573,2018,48,38.54,4,Donets'k
53574,2018,49,35.67,4,Donets'k
53575,2018,50,35.04,4,Donets'k
53576,2018,51,38.12,4,Donets'k


Пошук екстремумів (min та max) для вказаних областей та років, середнього, медіани

In [26]:
def filter_stats_provinces_years(df, provinces_ids, years):
    df_filter_py = df[(df["province_id"].isin(provinces_ids)) & (df["year"].isin(years))]
    stats = df_filter_py.groupby("province_name")["VHI"].agg(["min", "max", "mean", "median"])
    return stats

years = list(map(int, input("Введіть роки для виборки у форматі (yyyy yyyy yyyy): ").split()))
provinces = list(map(int, input("Введіть індекси областей для виборки у форматі (ii ii ii): ").split()))
filter_stats_provinces_years(df, provinces, years)

,min,max,mean,median
province_name,,,,
Cherkasy,27.18,70.83,47.482885,45.30
Donets'k,27.64,65.80,43.162500,41.52
Khmel'nyts'kyy,27.59,66.02,46.304744,47.10
